# 🟠 Kafka — Parte 1: Subindo Kafka Local e Criando Tópicos

---

> **Objetivo:** Instalar e iniciar um broker Kafka funcional dentro do Colab, explorar a CLI e criar tópicos com diferentes configurações.

| Etapa | O que faremos |
|---|---|
| **1. Instalação** | Baixar Kafka + Java no Colab |
| **2. Start do Broker** | Subir ZooKeeper e Kafka em background |
| **3. CLI — Tópicos** | Criar, listar, descrever e deletar tópicos |
| **4. Conceitos** | Partições, replication factor, retention |

---
## ⚙️ Etapa 1 — Instalação do Kafka

In [ ]:
# ============================================================
# ETAPA 1.1 — INSTALACAO
# ============================================================

import subprocess, os, time

KAFKA_VERSION = "3.7.0"
SCALA_VERSION = "2.13"
KAFKA_DIR     = f"/opt/kafka_{SCALA_VERSION}-{KAFKA_VERSION}"
KAFKA_URL     = f"https://archive.apache.org/dist/kafka/{KAFKA_VERSION}/kafka_{SCALA_VERSION}-{KAFKA_VERSION}.tgz"
KAFKA_ARCHIVE = "/tmp/kafka.tgz"
BIN           = f"{KAFKA_DIR}/bin"

# -- 1. Java --
print("Verificando / instalando Java...")
if subprocess.run(["java", "-version"], capture_output=True).returncode != 0:
    os.system("apt-get install -y -q default-jre-headless 2>/dev/null")
print("Java OK")

# -- 2. Download + extracao --
start_sh = f"{BIN}/zookeeper-server-start.sh"

if os.path.isfile(start_sh):
    print(f"Kafka ja instalado em {KAFKA_DIR}")
else:
    print(f"Baixando Kafka {KAFKA_VERSION} (~113 MB)...")
    os.system(f"rm -rf {KAFKA_DIR} {KAFKA_ARCHIVE}")

    ret = os.system(f"wget -q --show-progress {KAFKA_URL} -O {KAFKA_ARCHIVE}")

    if ret != 0 or not os.path.isfile(KAFKA_ARCHIVE):
        raise RuntimeError("Download falhou. Verifique a conexao.")

    size_mb = os.path.getsize(KAFKA_ARCHIVE) / (1024 * 1024)
    print(f"Arquivo baixado: {size_mb:.1f} MB")
    if size_mb < 50:
        raise RuntimeError(f"Arquivo muito pequeno ({size_mb:.1f} MB) — download incompleto.")

    print("Extraindo...")
    ret = os.system(f"tar -xzf {KAFKA_ARCHIVE} -C /opt/")
    if ret != 0:
        raise RuntimeError("Falha ao extrair o arquivo.")

    if not os.path.isfile(start_sh):
        raise RuntimeError(f"Extracao concluida mas {start_sh} nao encontrado.")

    print(f"Kafka extraido com sucesso em {KAFKA_DIR}")

print(f"\nScripts disponiveis em {BIN}/:")
scripts = [f for f in os.listdir(BIN) if f.endswith('.sh')]
print("   " + ", ".join(sorted(scripts)[:8]) + "...")

---
## 🚀 Etapa 2 — Subindo o Broker

In [ ]:
# ============================================================
# ETAPA 2.1 — INICIAR ZOOKEEPER
# ============================================================

import subprocess, os, time

KAFKA_DIR = "/opt/kafka_2.13-3.7.0"
BIN       = f"{KAFKA_DIR}/bin"
ZK_LOG    = "/tmp/zookeeper.log"

# ── Instalar netcat se ausente ──
print("Verificando netcat...")
if subprocess.run(["which", "nc"], capture_output=True).returncode != 0:
    os.system("apt-get install -y -q netcat-openbsd 2>/dev/null")
    print("   netcat instalado.")
else:
    print("   netcat já disponível.")

# ── Matar processos anteriores se houver ──
os.system("pkill -f zookeeper 2>/dev/null; sleep 1")

# ── Iniciar ZooKeeper ──
ZK_CMD = f"{BIN}/zookeeper-server-start.sh {KAFKA_DIR}/config/zookeeper.properties"
print("Iniciando ZooKeeper...")

zk_proc = subprocess.Popen(
    ZK_CMD.split(),
    stdout=open(ZK_LOG, 'w'),
    stderr=subprocess.STDOUT,
    preexec_fn=os.setsid   # Isola o processo para não morrer com o kernel do Colab
)

# ── Aguardar com polling ──
print("   Aguardando porta 2181", end="")
ok = False
for _ in range(20):           # Tenta por até 20s
    time.sleep(1)
    print(".", end="", flush=True)
    chk = subprocess.run(
        ["nc", "-z", "-w1", "localhost", "2181"],
        capture_output=True
    )
    if chk.returncode == 0:
        ok = True
        break

print()
if ok:
    print("✅ ZooKeeper rodando na porta 2181")
else:
    print("❌ ZooKeeper não respondeu em 20s. Últimas linhas do log:")
    os.system(f"tail -30 {ZK_LOG}")

In [ ]:
# ============================================================
# ETAPA 2.2 — INICIAR O BROKER KAFKA
# ============================================================

KAFKA_LOG = "/tmp/kafka.log"
KAFKA_CMD = f"{BIN}/kafka-server-start.sh {KAFKA_DIR}/config/server.properties"

print("Iniciando Kafka Broker...")
kafka_proc = subprocess.Popen(
    KAFKA_CMD.split(),
    stdout=open(KAFKA_LOG, 'w'),
    stderr=subprocess.STDOUT
)

time.sleep(8)  # Kafka demora um pouco mais que o ZK

result = subprocess.run(["nc", "-z", "localhost", "9092"], capture_output=True)
if result.returncode == 0:
    print("Kafka Broker rodando na porta 9092")
    print("\n Arquitetura ativa:")
    print("   [ZooKeeper :2181] ←coordenação→ [Kafka Broker :9092]")
    print("                                          ↑")
    print("                                   Producers & Consumers")
else:
    print("❌ Kafka não respondeu — verifique o log:")
    os.system(f"tail -30 {KAFKA_LOG}")

---
## 📋 Etapa 3 — CLI: Gerenciando Tópicos

In [ ]:
# ============================================================
# ETAPA 3.1 — CRIAR TÓPICOS COM DIFERENTES CONFIGURAÇÕES
# ============================================================
# Cada tópico tem duas configurações fundamentais:
#   --partitions       → quantas partições (paralelismo)
#   --replication-factor → quantas réplicas (tolerância a falhas)
#
# Como temos apenas 1 broker, replication-factor máximo = 1

import time, subprocess

BOOTSTRAP = "localhost:9092"
KAFKA_DIR = "/opt/kafka_2.13-3.7.0"
BIN       = f"{KAFKA_DIR}/bin"

def kafka_topics(action, **kwargs):
    cmd = [f"{BIN}/kafka-topics.sh", f"--{action}",
           "--bootstrap-server", BOOTSTRAP]
    for k, v in kwargs.items():
        cmd += [f"--{k.replace('_','-')}", str(v)]
    result = subprocess.run(cmd, capture_output=True, text=True)
    return result.stdout.strip() or result.stderr.strip()

def aguardar_broker_pronto(timeout=30):
    """
    Espera o broker estar pronto para operacoes administrativas.
    A porta 9092 pode estar aberta mas o broker ainda nao ter
    completado o registro no ZooKeeper e inicializado os
    controladores internos.
    """
    print("Aguardando broker estar pronto para operacoes...", end="")
    inicio = time.time()
    while time.time() - inicio < timeout:
        result = subprocess.run(
            [f"{BIN}/kafka-topics.sh", "--list",
             "--bootstrap-server", BOOTSTRAP],
            capture_output=True, text=True
        )
        if result.returncode == 0:
            print(" pronto.")
            return True
        time.sleep(2)
        print(".", end="", flush=True)
    print(" timeout.")
    return False

# Aguarda o broker aceitar operacoes antes de criar topicos
if not aguardar_broker_pronto():
    raise RuntimeError("Broker nao ficou pronto a tempo. Verifique o log: tail -30 /tmp/kafka.log")

# Criar topico simples — 1 particao
print("\nCriando topico simples: 'eventos-usuarios'")
print(kafka_topics("create", topic="eventos-usuarios",
                   partitions=1, replication_factor=1))

# Criar topico com multiplas particoes — maior throughput
print("\nCriando topico com 3 particoes: 'transacoes-financeiras'")
print(kafka_topics("create", topic="transacoes-financeiras",
                   partitions=3, replication_factor=1))

# Criar topico com retencao curta (1 hora) — ideal para alertas
print("\nCriando topico com retencao de 1h: 'alertas-sistema'")
cmd = [
    f"{BIN}/kafka-topics.sh", "--create",
    "--bootstrap-server", BOOTSTRAP,
    "--topic", "alertas-sistema",
    "--partitions", "1",
    "--replication-factor", "1",
    "--config", "retention.ms=3600000",
    "--config", "cleanup.policy=delete"
]
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout.strip() or result.stderr.strip())

print("\n✅ Topicos criados com sucesso!")

In [ ]:
# ============================================================
# ETAPA 3.2 — LISTAR TÓPICOS EXISTENTES
# ============================================================

print("Tópicos existentes no broker:")
print(kafka_topics("list"))

In [ ]:
# ============================================================
# ETAPA 3.3 — DESCREVER UM TÓPICO (partições, líderes, réplicas)
# ============================================================

"""
    Por que ISR importa:
    Se uma réplica fica muito atrasada em relação ao líder, ela sai do ISR.
    Uma escrita só é considerada confirmada quando todas as réplicas do ISR
    acusam o recebimento. Se o ISR encolher, o cluster fica mais vulnerável
    a perda de dados em caso de falha.
"""

print("Detalhes do tópico 'transacoes-financeiras':")
print(kafka_topics("describe", topic="transacoes-financeiras"))

print("\n Explicação das colunas:")
print("   Topic      → nome do tópico")
print("   Partition  → ID da partição (0, 1, 2...)")
print("   Leader     → broker responsável por essa partição")
print("   Replicas   → quais brokers têm cópia da partição")
print("   Isr        → In-Sync Replicas — réplicas atualizadas com o líder")



In [ ]:
# ============================================================
# ETAPA 3.4 — ALTERAR CONFIGURAÇÃO DE UM TÓPICO
# ============================================================
# Podemos alterar configs em runtime sem recriar o tópico!

print(" Alterando retenção de 'eventos-usuarios' para 7 dias...")
cmd = [
    f"{BIN}/kafka-configs.sh",
    "--bootstrap-server", BOOTSTRAP,
    "--entity-type", "topics",
    "--entity-name", "eventos-usuarios",
    "--alter",
    "--add-config", "retention.ms=604800000"  # 7 dias
]
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout.strip() or result.stderr.strip())

# Verificar configuração aplicada
print("\n Verificando configuração atual:")
cmd_desc = [
    f"{BIN}/kafka-configs.sh",
    "--bootstrap-server", BOOTSTRAP,
    "--entity-type", "topics",
    "--entity-name", "eventos-usuarios",
    "--describe"
]
result = subprocess.run(cmd_desc, capture_output=True, text=True)
print(result.stdout.strip())

In [ ]:
# ============================================================
# ETAPA 3.5 — AUMENTAR PARTIÇÕES (scale out)
# ============================================================
# Podemos AUMENTAR partições, mas nunca reduzir.
# Isso permite escalar o throughput sem recriar o tópico.

print(" Aumentando 'eventos-usuarios' de 1 para 4 partições...")
print(kafka_topics("alter", topic="eventos-usuarios", partitions=4))

print("\n Verificando nova configuração:")
print(kafka_topics("describe", topic="eventos-usuarios"))

In [ ]:
# ============================================================
# ETAPA 3.6 — DELETAR TÓPICO
# ============================================================

print(" Deletando tópico 'alertas-sistema'...")
print(kafka_topics("delete", topic="alertas-sistema"))

time.sleep(2)
print("\nTópicos restantes:")
print(kafka_topics("list"))

---
## 📚 Etapa 4 — Conceitos Fundamentais

### Partições
- Cada tópico é dividido em **N partições** — unidades de paralelismo
- Mensagens com a **mesma key** sempre vão para a **mesma partição** (ordem garantida)
- Consumidores em um grupo processam **partições diferentes em paralelo**

### Replication Factor
- Quantas cópias de cada partição existem no cluster
- `replication-factor=3` significa que 2 brokers podem cair sem perda de dados
- Em produção: sempre ≥ 3

### Retention
- O Kafka **não deleta mensagens após consumo** — elas ficam retidas por tempo ou tamanho
- `retention.ms=604800000` → 7 dias (padrão)
- `retention.bytes=-1` → sem limite de tamanho (padrão)

```
Topic: transacoes-financeiras
┌─────────────────────────────────────────┐
│  Partition 0: [msg0] [msg3] [msg6] ...  │
│  Partition 1: [msg1] [msg4] [msg7] ...  │
│  Partition 2: [msg2] [msg5] [msg8] ...  │
└─────────────────────────────────────────┘
        ↑ Offset cresce sempre (imutável)
```